In [3]:
from __future__ import annotations

import math
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "impl").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from impl.gaussian_baseline_quantizers import (
    CubicE8Quantizer,
    QuipSharpE8Quantizer,
    UniformScalarQuantizer,
    mse_to_sqnr_bits,
)
from impl.leech_lattice_vector_quantizer import DIM, LeechLatticeVectorQuantizer, squared_distance

print(ROOT)


/home/diego/Documents/cvlab/llvq-paper-reproduction


In [4]:
SEED = 0
N_SAMPLES = 8

# Cumulative Leech ball cutoffs. m=51 reaches 3.0 bits/dim.
SHELL_CUTOFFS = [2, 3, 4, 5, 6, 8, 10, 13, 20, 30, 40, 51]

# Baseline rates. Integer bits keep the simple baselines honest and fast.
BASELINE_RATES = [1.0, 2.0, 3.0]

# Idealistic Gaussian-source scaling matches source power E||w||^2 = DIM
# to the average squared norm of the finite Leech ball codebook.
GAUSSIAN_SOURCE_VARIANCE = 1.0

rng = np.random.default_rng(SEED)
samples = rng.normal(0.0, 1.0, size=(N_SAMPLES, DIM))
samples.shape


(8, 24)

In [ ]:
q = LeechLatticeVectorQuantizer(max_shell=14, verbose=True)
w = samples[0]

start = time.perf_counter()
idx = q.quantize(w)
quantize_seconds = time.perf_counter() - start

ranked = q.unrank(idx)

start = time.perf_counter()
integer_representative = q.dequantize_lattice(idx)
dequantize_lattice_seconds = time.perf_counter() - start

start = time.perf_counter()
scaled_reconstruction = q.dequantize(idx)
dequantize_seconds = time.perf_counter() - start

print("input:", np.round(w, 4))
print("index:", idx)
print("address:", ranked)
print("integer representative:", integer_representative)
print("scaled reconstruction:", np.round(scaled_reconstruction, 4))
print("block MSE:", squared_distance(w, scaled_reconstruction) / DIM)
print(f"quantize time: {quantize_seconds:.6f}s")
print(f"dequantize_lattice time: {dequantize_lattice_seconds:.6f}s")
print(f"dequantize time: {dequantize_seconds:.6f}s")

loaded cached class leaders for shell m=2: 3 classes in 0.000s
loaded cached class leaders for shell m=3: 4 classes in 0.000s
loaded cached class leaders for shell m=4: 8 classes in 0.000s
loaded cached class leaders for shell m=5: 9 classes in 0.000s
loaded cached class leaders for shell m=6: 17 classes in 0.000s
loaded cached class leaders for shell m=7: 17 classes in 0.000s
loaded cached class leaders for shell m=8: 30 classes in 0.000s
loaded cached class leaders for shell m=9: 33 classes in 0.000s
loaded cached class leaders for shell m=10: 47 classes in 0.000s
loaded cached class leaders for shell m=11: 54 classes in 0.000s
loaded cached class leaders for shell m=12: 79 classes in 0.000s
loaded cached class leaders for shell m=13: 82 classes in 0.000s
loaded cached class leaders for shell m=14: 115 classes in 0.000s
input: [ 0.1257 -0.1321  0.6404  0.1049 -0.5357  0.3616  1.304   0.9471 -0.7037
 -1.2654 -0.6233  0.0413 -2.325  -0.2188 -1.2459 -0.7323 -0.5443 -0.3163
  0.4116  1.0